In [68]:
import re
from rapidfuzz import fuzz
from transformers import pipeline

In [69]:
def fuzzy_check(text):
    # targets = ["kill", "suicide", "rape", "nude"]
    targets = []

    for word in targets:
        score = max(
            fuzz.partial_ratio(text, word),
            fuzz.ratio(text, word)
        )

        if score > 80:
            return True, word

    return False, None

In [70]:
# Model 1 → toxicity (insults)
model_1 = pipeline(
    "text-classification",
    model="unitary/unbiased-toxic-roberta"
)

# Model 3 → multi-label risk
model_3 = pipeline(
    "text-classification",
    model="KoalaAI/Text-Moderation"
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 30580.93it/s]
RobertaForSequenceClassification LOAD REPORT from: unitary/unbiased-toxic-roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 6350.10it/s]
DebertaForSequenceClassification LOAD REPORT from: KoalaAI/Text-Moderation
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [71]:
def normalize_text(text: str) -> str:
    text = text.lower()

    replacements = {
        '3': 'e', '4': 'a', '5': 's',
        '0': 'o', '@': 'a', '$': 's',
        '1': 'l'
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    text = re.sub(r'[^a-z0-9\s]', '', text)

    # merge spaced letters: k i l l → kill
    text = re.sub(r'(\b[a-z]\s+){2,}[a-z]\b',
                  lambda m: m.group(0).replace(" ", ""),
                  text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [72]:
def normalize_text(text: str) -> str:
    text = text.lower()

    replacements = {
        '3': 'e', '4': 'a', '5': 's',
        '0': 'o', '@': 'a', '$': 's',
        '1': 'l'
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    text = re.sub(r'[^a-z0-9\s]', '', text)

    # merge spaced letters: k i l l → kill
    text = re.sub(r'(\b[a-z]\s+){2,}[a-z]\b',
                  lambda m: m.group(0).replace(" ", ""),
                  text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [73]:
def predict_model_1(text):
    output = model_1(text)[0]

    label = output["label"]
    score = output["score"]

    # LABEL_1 = non-toxic → invert
    if label == "LABEL_1":
        return 1 - score
    else:
        return score

In [74]:
def predict_model_3(text):
    output = model_3(text, top_k=None)

    if isinstance(output[0], list):
        output = output[0]

    return {o["label"]: o["score"] for o in output}

In [75]:
def detect_grooming(messages):
    risk_score = 0

    for msg in messages:
        msg = msg.lower()

        if any(x in msg for x in ["meet", "come over", "visit"]):
            risk_score += 1

        if any(x in msg for x in ["dont tell", "secret", "keep this"]):
            risk_score += 2

        if any(x in msg for x in ["alone", "private", "just us"]):
            risk_score += 2

    return risk_score >= 3

In [76]:
SAFE_PHRASES = [
    "kill time",
    "kill the process",
    "kill the app",
    "kill a task"
]

def is_safe_context(text):
    for phrase in SAFE_PHRASES:
        if phrase in text:
            return True
    return False

In [77]:
def interpret_scores(scores):
    return {
        "safe": scores.get("OK", 0),
        "violence": max(scores.get("V", 0), scores.get("V2", 0)),
        "hate": max(scores.get("H", 0), scores.get("H2", 0), scores.get("HR", 0)),
        "sexual": max(scores.get("S", 0), scores.get("S3", 0)),
        "self_harm": scores.get("SH", 0)
    }

In [78]:
# def decide(text, scores_m3, toxicity_score):

#     # ✅ safe override
#     if is_safe_context(text):
#         return "APPROVED", "safe_context"

#     # 🚨 threat detection
#     if scores_m3.get("threat", 0) > 0.4:
#         return "BLOCKED", "threat"

#     if scores_m3.get("dangerous", 0) > 0.6:
#         return "BLOCKED", "dangerous"

#     # ⚠️ toxicity (Model 1)
#     if toxicity_score > 0.8:
#         return "BLOCKED", "toxicity"

#     if toxicity_score > 0.5:
#         return "FLAGGED", "toxicity"

#     # ⚠️ moderate danger
#     if scores_m3.get("dangerous", 0) > 0.3:
#         return "FLAGGED", "dangerous"

#     return "APPROVED", None

def decide_from_codes(scores):

    s = interpret_scores(scores)

    # 🚨 BLOCK
    if s["violence"] > 0.5:
        return "BLOCKED", "violence"

    if s["self_harm"] > 0.5:
        return "BLOCKED", "self_harm"

    if s["sexual"] > 0.6:
        return "BLOCKED", "sexual"

    # ⚠️ FLAG
    if s["hate"] > 0.4:
        return "FLAGGED", "hate"

    # ✅ SAFE
    return "APPROVED", None

In [79]:
def moderate(messages):
    latest = messages[-1]

    processed = normalize_text(latest)

    # 🔹 Fuzzy check
    fuzzy_hit, word = fuzzy_check(processed)
    if fuzzy_hit:
        return {
            "status": "BLOCKED",
            "reason": f"fuzzy_match:{word}"
        }

    # 🔹 Context check (grooming)
    if detect_grooming(messages):
        return {
            "status": "BLOCKED",
            "reason": "grooming_risk"
        }

    # 🔹 Model predictions
    toxicity_score = predict_model_1(processed)
    scores_m3 = predict_model_3(processed)

    status, reason = decide(processed, scores_m3, toxicity_score)

    return {
        "text": latest,
        "processed": processed,
        "toxicity_score": toxicity_score,
        "model3_scores": scores_m3,
        "status": status,
        "reason": reason
    }

In [80]:
tests = [
    ["Hello, how are you?"],
    ["I hate you"],
    ["You are stupid"],
    ["Send me nudes"],
    ["I will kill you"],
    ["k i l l"],
    ["k!ll"],
    ["ki11"],
    ["Let's build something amazing"],
    ["You are annoying"],
    ["Go to hell"],
    ["Lets meet tomorrow","kill some one"],
    ["lets meet tomorrow", "dont tell your parents"]
]

for t in tests:
    print("INPUT:", t)
    print(moderate(t))
    print("-" * 80)

INPUT: ['Hello, how are you?']
{'text': 'Hello, how are you?', 'processed': 'hello how are you', 'toxicity_score': 0.007151651196181774, 'model3_scores': {'OK': 0.9493230581283569, 'H': 0.020667454227805138, 'V': 0.007245813962072134, 'SH': 0.007077094167470932, 'HR': 0.005866572260856628, 'S': 0.0035704507026821375, 'V2': 0.0025019189342856407, 'S3': 0.002107697306200862, 'H2': 0.0016399569576606154}, 'status': 'APPROVED', 'reason': None}
--------------------------------------------------------------------------------
INPUT: ['I hate you']
{'text': 'I hate you', 'processed': 'i hate you', 'toxicity_score': 0.8707150220870972, 'model3_scores': {'SH': 0.29994386434555054, 'H': 0.16168217360973358, 'V': 0.1520620882511139, 'HR': 0.13304388523101807, 'OK': 0.10772812366485596, 'H2': 0.05988021194934845, 'V2': 0.0408688448369503, 'S': 0.028293117880821228, 'S3': 0.016497664153575897}, 'status': 'BLOCKED', 'reason': 'toxicity'}
---------------------------------------------------------------